# 21. 최종 모델 주변 비교 — 변수 조합 / 비지도 모델

- **A. 모델 고정(Mahalanobis) × 변수 조합 12개**: 원값 vs 파생, 그룹 단독, 그룹 하나씩 빼기, 더하기
- **B. 변수 고정(free 17개) × 비지도 모델 10개**: maha, 축소 없는 maha, kNN, LOF, iForest, PCA, OCSVM, GMM, 오토인코더, 단순 최대편차

공통: 같은 reference(라벨 미사용)로 학습, 같은 labeled로 테스트, 하이퍼파라미터 사전 고정(튜닝 없음)
**기준 = free · maha (최종 모델)** → 각 방법과의 AP 차이를 paired bootstrap 95% CI로 판정

⚠ 같은 테스트셋에서 여러 개를 비교해 1등을 고르면 '선택 과적합' → **기준보다 CI상 뚜렷이 좋을 때만** 교체 검토

In [ ]:
import os
import pandas as pd
from compare_final import prepare, run_compare, plot_ap, COMBOS, MODEL_PARAMS
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 40)
OUT = './outputs/compare_final'
os.makedirs(OUT, exist_ok=True)

DATA = {p: prepare(p) for p in ['cn7', 'rg3']}      # reference·labeled 파생변수 (제품당 한 번)

In [ ]:
A, B = run_compare(DATA)                             # 수 분 소요 (bootstrap 포함)
A.to_csv(f'{OUT}/A_combos.csv', index=False, encoding='utf-8-sig')
B.to_csv(f'{OUT}/B_models.csv', index=False, encoding='utf-8-sig')

## A. 변수 조합 (모델 = Mahalanobis 고정)
- 기준(free) 행이 20b의 결과(CN7 ① 0.373 / ② 0.212)와 같은지 먼저 확인
- **판정**: 기준 대비 ΔAP의 95% CI가 0을 안 걸치면 '좋음/나쁨', 걸치면 '차이 없음'

In [ ]:
cols = ['제품', '변수 조합', '변수 수', '①AP', '①유의', '①상위10%', '①판정', '②AP', '②AP_lo', '②AP_hi',
        '②Run별 상위10%', '②AP_가스', '②AP_미성형', '②ΔAP(vs 기준)', '②Δ CI', '②판정', 'q0.99 경보비율', 'q0.999 TP', 'q0.999 경보']
for p in ['cn7', 'rg3']:
    print(p.upper())
    display(A[A['제품'] == p][cols].sort_values('②AP', ascending=False).round(3))

In [ ]:
for p in ['cn7', 'rg3']:
    plot_ap(A, '변수 조합', p, '변수 조합 비교 (모델 = Mahalanobis)', save=f'{OUT}/A_{p}.png')

## B. 비지도 모델 (변수 = free 17개 고정)
- **q0.99 경보비율**: 1%에 가까울수록 임계값 보정이 잘 된 것 (reference 점수를 자기 자신 포함으로 계산하는 모델은 조금 높게 나올 수 있음)

In [ ]:
cols = ['제품', '모델', '설명', '①AP', '①유의', '①상위10%', '①판정', '②AP', '②AP_lo', '②AP_hi',
        '②Run별 상위10%', '②AP_가스', '②AP_미성형', '②ΔAP(vs 기준)', '②Δ CI', '②판정', 'q0.99 경보비율', 'q0.999 TP', 'q0.999 경보']
for p in ['cn7', 'rg3']:
    print(p.upper())
    display(B[B['제품'] == p][cols].sort_values('②AP', ascending=False).round(3))

In [ ]:
for p in ['cn7', 'rg3']:
    plot_ap(B, '모델', p, '비지도 모델 비교 (변수 = free)', save=f'{OUT}/B_{p}.png')

## 해석 가이드
- **A**
  - `raw_all`·`level` vs `free` → 파생변수(시계열 편차)의 효과
  - `free − X`에서 AP가 크게 떨어지는 그룹 = 성능을 만드는 그룹 / 거의 안 변하면 기여 작음
  - `free + delta/level`이 안 좋아지면 → 변수를 더한다고 좋아지지 않음 (현재 세트가 적정)
- **B**
  - 상위권이 maha·pca·maha_emp처럼 '공분산(선형 상관)' 계열이면 → 변수 간 관계가 깨지는 것이 불량 신호라는 해석 지지
  - `maxdev`(학습 없음)보다 낫다 = 모델이 변수 간 관계를 학습한 효과
- **최종 결정**: 기준보다 '좋음' 판정이 ①·② 모두에서 나온 방법만 교체 후보. 아니면 최종 모델 유지 + 이 표를 '견고성 근거'로 보고서에